##### D004_병합집계_보고서
# [모두마켓] 분기 성과 요약 보고서

## 1. 데이터 준비

In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas  -q

import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.5.4
pandas: 3.0.6


In [2]:
# ─────────────────────────────────────────────
# 종합 실습용 데이터 — 새 스냅샷 (이 셀부터 단독 실행 가능)
# ─────────────────────────────────────────────
np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})
print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

스냅샷 준비 완료 — orders: (1500, 6) | customers: (200, 3) | products: (30, 3)


## 2. 키 검증(필수)
- 병합 전 키가 유니크하고 빈 값이 아닌지 검증해야한다.
    - merge는 키에 문제가 있어도 오류 없이 조용히 잘못된 결과를 내기 때문!
        - 예: 행 폭증, 조용한 결측
- '1. 키가 중복되는가, 2. 짝이 안 맞는 키가 있는가, 3. 합친 뒤 행 수가 예상과 같은가'를 확인해야함
        

In [3]:
print("[병합 전 키 검증]")
# (1) 중복
print("ordr 키 중복:", ordr["order_id"].duplicated().sum(), "건")
print("cust 키 중복:", cust["customer_id"].duplicated().sum(), "건")
print("prod 키 중복:", prod["product_id"].duplicated().sum(), "건")
# (2) 미매칭
print("미매칭 customer_id:", (~ordr["customer_id"].isin(cust["customer_id"])).sum(), "건")
print("미매칭 product_id :", (~ordr["product_id"].isin(prod["product_id"])).sum(), "건")
# (3) 관계: m:1이면 병합 후 행 수 = 주문 수
expected_rows = len(ordr)
print("병합 후 예상 행 수:", expected_rows)

[병합 전 키 검증]
ordr 키 중복: 0 건
cust 키 중복: 0 건
prod 키 중복: 0 건
미매칭 customer_id: 0 건
미매칭 product_id : 0 건
병합 후 예상 행 수: 1500


검증결과 orders, customers, products 키 중복 0건이고, orders의 customer_id, product_id 미매칭 0건이므로 m:1이 병합이 안전하다.

## 3. 병합

In [4]:
# (2) 병합: validate로 관계 가정(m:1)을 강제. indicator로 매칭 확인.
df = (
    ordr
    .merge(cust, on="customer_id", how="left", validate="m:1")
    .merge(prod, on="product_id", how="left", validate="m:1")
)
print("\n[병합 결과] 행 수:", len(df), "(원본 주문 수와 같으면 폭증 없음)")
display(df.head(3))


[병합 결과] 행 수: 1500 (원본 주문 수와 같으면 폭증 없음)


,order_id,customer_id,product_id,quantity,amount,order_date,region,membership,category,price
0,O00001,C0032,P005,3,120000.0,2025-04-12,서울,premium,패션,40000
1,O00002,C0049,P020,1,25000.0,2025-03-09,서울,basic,식품,25000
2,O00003,C0041,P016,3,36000.0,2025-01-27,인천,basic,패션,12000


## 4. 집계

#### 4.1 지역 × 회원등급 매출 교차표

In [5]:
region_membership = pd.pivot_table(
    df, index="region", columns="membership", values="amount", aggfunc="sum", fill_value=0,
    margins=True, margins_name="합계"
)
print("\n[지역 × 회원등급 매출 교차표]")
display(region_membership)


[지역 × 회원등급 매출 교차표]


membership,basic,premium,vip,합계
region,,,,
경기,14594000.0,3226000.0,3325000.0,21145000.0
부산,11531000.0,4857000.0,2366000.0,18754000.0
서울,11914000.0,9244000.0,2322000.0,23480000.0
인천,17372000.0,9675000.0,530000.0,27577000.0
합계,55411000.0,27002000.0,8543000.0,90956000.0


In [6]:
# 지역별 매출 비중 + 지역 내 vip 비중 (보고서 발견 1 근거)
total = region_membership.loc["합계", "합계"]
region_ratio = pd.DataFrame({
    "매출": region_membership["합계"],
    "전체 대비(%)": (region_membership["합계"] / total * 100).round(1),
    "vip 매출": region_membership["vip"],
    "지역 내 vip 비중(%)": (region_membership["vip"] / region_membership["합계"] * 100).round(1),
}).drop(index="합계")
print("[지역별 매출 비중 & 지역 내 vip 비중]")
display(region_ratio.sort_values("매출", ascending=False))

# 회원등급별 전체 매출 비중 (보고서 발견 3 근거)
print("\n[회원등급별 매출 비중(%)]")
display((region_membership.loc["합계"] / total * 100).round(1).drop("합계"))

[지역별 매출 비중 & 지역 내 vip 비중]


,매출,전체 대비(%),vip 매출,지역 내 vip 비중(%)
region,,,,
인천,27577000.0,30.3,530000.0,1.9
서울,23480000.0,25.8,2322000.0,9.9
경기,21145000.0,23.2,3325000.0,15.7
부산,18754000.0,20.6,2366000.0,12.6



[회원등급별 매출 비중(%)]


membership
basic      60.9
premium    29.7
vip         9.4
Name: 합계, dtype: float64

#### 4.2 카테고리별 KPI(매출,건수,객단가) 요약표

In [7]:
category_kpi = (
    df.groupby("category")
    .agg(
        매출=("amount", "sum"),
        건수=("order_id", "count"),
        객단가=("amount", "mean"),
    )
    .sort_values("매출", ascending=False)
    .round(0)
)

# 카테고리별 매출 비중 + 전체 객단가 (보고서 발견 2·3 근거)
category_kpi["매출 비중(%)"] = (category_kpi["매출"] / category_kpi["매출"].sum() * 100).round(1)
print("[카테고리별 KPI(매출,건수,객단가) + 매출 비중 요약표]")
display(category_kpi)
print(f"\n전체 객단가: {df['amount'].mean():,.0f}원")

[카테고리별 KPI(매출,건수,객단가) + 매출 비중 요약표]


,매출,건수,객단가,매출 비중(%)
category,,,,
패션,38318000.0,512,74840.0,42.1
식품,22404000.0,398,56291.0,24.6
가전,20248000.0,398,50874.0,22.3
뷰티,9986000.0,192,52010.0,11.0



전체 객단가: 60,637원


## 5. 특성 추가

In [8]:
df["region_total"] = df.groupby("region")["amount"].transform("sum")
df["share_in_region(%)"] = (df["amount"] / df["region_total"] * 100).round(3)
print("\n[주문별 해당 지역 매출 점유율 - 앞 5행]")
display(df[["order_id", "region", "amount", "region_total", "share_in_region(%)"]].head())


[주문별 해당 지역 매출 점유율 - 앞 5행]


,order_id,region,amount,region_total,share_in_region(%)
0,O00001,서울,120000.0,23480000.0,0.511
1,O00002,서울,25000.0,23480000.0,0.106
2,O00003,인천,36000.0,27577000.0,0.131
3,O00004,경기,120000.0,21145000.0,0.568
4,O00005,경기,120000.0,21145000.0,0.568


In [9]:
# 지역별 주문 1건의 최대 매출 비중(%) — 특정 주문에 매출이 몰려 있는지 확인
print("[지역별 주문 1건 최대 비중(%)]")
display(df.groupby("region")["share_in_region(%)"].max())

[지역별 주문 1건 최대 비중(%)]


region
경기    1.064
부산    1.200
서울    0.958
인천    0.816
Name: share_in_region(%), dtype: float64

## 6. 보고서

### 1. 분석 개요
- 대상: 모두마켓 주문 1,500건 (2025-01 ~ 2025-05), 고객 200명·상품 30개 정보 병합
- 방법: 3개 테이블 키 검증 후 `m:1` 병합 → 지역×회원등급 매출 교차표(`pivot_table`) → 카테고리별 KPI(`groupby`+`agg`) → 주문별 지역 매출 비중(`transform`)

### 2. 핵심 발견
- **[지역 × 회원등급 교차표]** 인천이 매출 2,757.7만 원(전체의 30.3%)으로 1위지만, 인천의 vip 매출은 53만 원(지역 매출의 1.9%)으로 4개 지역 중 가장 낮음(경기 15.7%, 부산 12.6%, 서울 9.9%). 전체로 봐도 매출의 60.9%가 basic, vip는 9.4%에 그쳐 basic 고객 의존도가 높음.
- **[카테고리별 KPI]** 패션이 매출 3,831.8만 원(42.1%)으로 1위이며, 건수(512건)와 객단가(74,840원)도 모두 가장 높음. 많이 팔리고 단가도 높은 핵심 카테고리임. 반면 뷰티는 매출 비중 11.0%(192건)로 가장 낮음. (전체 객단가 약 60,637원)
- **[특성 추가: 지역 내 주문 비중]** 주문 1건이 지역 매출에서 차지하는 비중은 최대 1.2%(부산), 인천은 0.8%로 특정 주문에 매출이 몰려 있지 않음. 지역 매출은 소수의 큰 주문이 아니라 다수 주문이 고르게 쌓인 결과이며, 매출 규모가 작은 부산일수록 주문 1건의 영향이 큼.

### 3. 데이터 신뢰성
- 병합 전 키 검증: 3개 테이블 키 중복 0건, 미매칭 0건 → 행 폭증·조용한 결측 없음.
- `validate="m:1"` 통과 → 관계 가정(주문:고객, 주문:상품 = m:1) 유효. 병합 후 1,500행으로 원본 주문 수와 같음.

### 4. 제언
- 매출 1위인 인천은 vip 비중(1.9%)이 가장 낮으므로, 인천의 basic·premium 고객을 대상으로 매출·객단가 1위인 **패션 카테고리 중심의 vip 전환 프로모션**을 진행해 지역 고객 가치를 높인다.